# Benchmark Quantum Simulator en AMD ROCm

Ejecuta el benchmark del simulador statevector en GPU AMD (W7900D, RDNA 3, gfx1100) y genera archivos de resultados.

**Flujo:**
1. Clona repo → 2. Busca directorio que contenga AMBOS `qsim_torch.py` Y `hardware_wheels.py` → 3. Instala deps → 4. Detecta hardware → 5. Ejecuta benchmark → 6. Guarda resultados.

**IMPORTANTE:** Guarda los archivos generados en `/persistent` o `/workspace` según tu URL de sesión para que sobrevivan al apagar la sesión.

**No hardcodea rutas de usuario ni nombres de subdirectorios.** Todo se descubre en runtime.

In [ ]:
# ============================================================================
# CELDA 1: Clonar repositorio y descubrir directorio del simulador
# ============================================================================
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/jalp17/chern-calibration-copilot.git"
REPO_NAME = "chern-calibration-copilot"

print(f"Directorio actual: {Path.cwd()}")

repo_dir = Path.cwd() / REPO_NAME

if repo_dir.exists() and (repo_dir / ".git").exists():
    print(f"Repo ya existe en {repo_dir}. Actualizando...")
    subprocess.run(["git", "-C", str(repo_dir), "pull"], capture_output=True, text=True)
else:
    print(f"Clonando {REPO_URL} ...")
    result = subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"ERROR clone: {result.stderr}")
        raise SystemExit(1)
    print("✓ Repo clonado")

# ============================================================================
# PASO 1b: Buscar directorio que contenga AMBOS archivos clave
# ============================================================================

SIM_DIR = None
for p in repo_dir.rglob("qsim_torch.py"):
    candidato = p.parent
    if (candidato / "hardware_wheels.py").exists():
        SIM_DIR = candidato.resolve()
        print(f"✓ Encontrado directorio con ambos archivos: {SIM_DIR}")
        break
else:
    # Fallback: buscar hardware_wheels.py directamente
    for p in repo_dir.rglob("hardware_wheels.py"):
        SIM_DIR = p.parent.resolve()
        print(f"✓ Encontrado hardware_wheels.py en: {SIM_DIR}")
        if (SIM_DIR / "qsim_torch.py").exists():
            print(f"  (también tiene qsim_torch.py)")
        break
    else:
        raise SystemExit(f"ERROR: No se encontró hardware_wheels.py en {repo_dir}")

print(f"\nSIM_DIR = {SIM_DIR}")

# FORZAR ruta absoluta en sys.path (PRIORIDAD MÁXIMA, índice 0)
sys.path.insert(0, str(SIM_DIR))
print(f"sys.path[0] = {sys.path[0]}")

In [ ]:
# ============================================================================
# CELDA 2: Instalar dependencias
# ============================================================================
import subprocess
import sys
from pathlib import Path

# Buscar requirements.txt (en directorio actual o padre)
req_file = Path("requirements.txt")
if not req_file.exists():
    reqs = list(Path(".").parent.rglob("requirements.txt"))
    if reqs:
        req_file = reqs[0]
        print(f"requirements.txt encontrado en: {req_file.parent}")
        os.chdir(str(req_file.parent))
    else:
        raise SystemExit("ERROR: No se encontró requirements.txt")

print(f"Instalando dependencias desde {req_file}...")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(req_file)], check=True)

# scikit-learn necesario para kernel_cuantico.py
subprocess.run([sys.executable, "-m", "pip", "install", "scikit-learn"], check=True)
print("✓ Dependencias instaladas")

In [ ]:
# ============================================================================
# CELDA 3: Detectar hardware (RUTA ABSOLUTA FORZADA EN sys.path)
# ============================================================================
import sys

# FORZAR ruta absoluta del simulador en sys.path (PRIORIDAD MÁXIMA, índice 0)
sys.path.insert(0, SIM_DIR)
print(f"SIM_DIR = {SIM_DIR}")
print(f"sys.path[0] = {sys.path[0]}")

# IMPORT DIRECTO - sin chdir, sin trucos
from hardware_wheels import detect_gpu_architecture, install_command, quick_summary

print("=" * 70)
print("DETECCIÓN DE HARDWARE")
print("=" * 70)
gpu = detect_gpu_architecture()
print(f"GPU             : {gpu.name}")
print(f"Arquitectura    : {gpu.architecture}")
print(f"ROCm            : {gpu.rocm_version}")
print(f"gfx1100         : {gpu.is_gfx1100}")
print(f"Wheel precompilada: {'SÍ' if gpu.supports_precompiled else 'NO'}")
print(f"\nComando install: {install_command(gpu)}")

In [ ]:
# ============================================================================
# CELDA 4: Ejecutar benchmark
# ============================================================================
from benchmark_notebook import run_benchmark

resultados = run_benchmark(
    device_str='cuda',
    ns=[16, 20, 24, 26, 28],
    batches=[1, 256, 1024],
    dtypes=None,
    output_dir='.'
)

In [ ]:
# ============================================================================
# CELDA 5: Resultados
# ============================================================================
from pathlib import Path

out = Path(resultados['output_files']['json'])
md = Path(resultados['output_files']['markdown'])
print(f"JSON: {out} ({out.stat().st_size/1024:.1f} KB)")
print(f"MD:   {md} ({md.stat().st_size/1024:.1f} KB)")
print(md.read_text())

In [ ]:
# Diagnóstico GPU (opcional)
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")
    print(f"ROCm: {torch.version.hip}")
else:
    print("CPU only")